# Data dictionary and BUC dimensions

Column meanings, the four required mapping checks and the final dimension list.

In [1]:
import pandas as pd
from aoi import load_data, load_hierarchies, check_hierarchies

df = load_data("M26 DA A2/M26_DA_A2_Part2.csv")
print(f"{len(df):,} records, {df.shape[1]} columns")

299,285 records, 42 columns


## 1. What does each column mean?

In [2]:
dictionary_rows = [
    ('age', 'int', 'Age in years.'),
    ('class_of_worker', 'categorical', 'Private, government, self-employed or other worker class.'),
    ('industry_code_detailed', 'integer code', 'Detailed industry code.'),
    ('occupation_code_detailed', 'integer code', 'Detailed occupation code.'),
    ('education', 'categorical', 'Highest education level.'),
    ('wage_per_hour', 'int', 'Hourly wage; units not verified.'),
    ('enrolled_in_edu_last_wk', 'categorical', 'Education enrollment during the previous week.'),
    ('marital_status', 'categorical', 'Marital status.'),
    ('industry_code_major', 'categorical', 'Broad industry group.'),
    ('occupation_code_major', 'categorical', 'Broad occupation group.'),
    ('race', 'categorical', 'Reported race category.'),
    ('hispanic_origin', 'categorical', 'Reported Hispanic origin.'),
    ('sex', 'categorical', 'Reported sex.'),
    ('labor_union_member', 'categorical', 'Union membership.'),
    ('unemployment_reason', 'categorical', 'Reason for unemployment, where applicable.'),
    ('employment_status', 'categorical', 'Employment status or work schedule.'),
    ('capital_gains', 'int', 'Capital gains amount.'),
    ('capital_losses', 'int', 'Capital losses amount.'),
    ('dividends', 'int', 'Dividend income amount.'),
    ('tax_filer_status', 'categorical', 'Tax filing status.'),
    ('region_prev_residence', 'categorical', 'Previous region, as recorded in the dataset.'),
    ('state_prev_residence', 'categorical', 'Previous state, as recorded in the dataset.'),
    ('household_family_status', 'categorical', 'Detailed household or family relationship.'),
    ('household_summary', 'categorical', 'Broad household relationship.'),
    ('instance_weight', 'float', 'Sampling weight; excluded from dimensions.'),
    ('migration_msa', 'categorical', 'Migration between metropolitan statistical areas.'),
    ('migration_reg', 'categorical', 'Migration between broad geographic regions.'),
    ('migration_within_reg', 'categorical', 'Migration within the same broad region.'),
    ('live_here_1_year_ago', 'categorical', 'Whether the person lived at the same address one year earlier.'),
    ('migration_sunbelt', 'categorical', 'Migration classification involving the Sunbelt region.'),
    ('num_persons_worked_for_employer', 'int', 'Employer count/code (0–6); exact meaning unverified.'),
    ('family_members_under_18', 'categorical', 'Parents present for persons under 18, or Not in universe.'),
    ('country_birth_father', 'categorical', "Father's country of birth."),
    ('country_birth_mother', 'categorical', "Mother's country of birth."),
    ('country_birth_self', 'categorical', 'Country of birth.'),
    ('citizenship', 'categorical', 'Citizenship or nativity classification.'),
    ('own_business_self_employed', 'int', 'Business/self-employment code (0, 1, 2), not binary.'),
    ('vet_questionnaire', 'categorical', 'Veterans questionnaire or service-related status.'),
    ('veterans_benefits', 'int', 'Veterans benefits classification/code.'),
    ('weeks_worked_in_year', 'int', 'Number of weeks worked during the year.'),
    ('year', 'int', 'Survey year: 94 or 95.'),
    ('income', 'categorical', 'Income class: <=50K or >50K.'),
]

data_dictionary = pd.DataFrame(dictionary_rows, columns=["column", "type", "meaning"])
data_dictionary["distinct_values"] = data_dictionary["column"].map(df.nunique())
data_dictionary["unknown_count"] = data_dictionary["column"].map(df.eq("Unknown").sum())

assert data_dictionary["column"].tolist() == df.columns.tolist()
data_dictionary

,column,type,meaning,distinct_values,unknown_count
0,age,int,Age in years.,91,0
1,class_of_worker,categorical,"Private, government, self-employed or other wo...",9,0
2,industry_code_detailed,integer code,Detailed industry code.,52,0
3,occupation_code_detailed,integer code,Detailed occupation code.,47,0
4,education,categorical,Highest education level.,17,0
5,wage_per_hour,int,Hourly wage; units not verified.,1425,0
6,enrolled_in_edu_last_wk,categorical,Education enrollment during the previous week.,3,0
7,marital_status,categorical,Marital status.,7,0
8,industry_code_major,categorical,Broad industry group.,24,0
9,occupation_code_major,categorical,Broad occupation group.,15,0


`?`, `NA` and blanks become `Unknown`; `Not in universe` stays separate. Repeated records are retained. Numeric category codes are labels, not quantities.

## 2. Do detailed values map to one summary?

Check the four supplied pairs against `hierarchies.json`. Values with multiple summaries are reported separately.

In [3]:
hierarchies = load_hierarchies("hierarchies.json")
mapping_checks = check_hierarchies(df, hierarchies)
mapping_checks[["source", "summary", "ambiguous_values", "ambiguous_rows", "mismatches"]]

,source,summary,ambiguous_values,ambiguous_rows,mismatches
0,industry_code_detailed,industry_code_major,0,0,0
1,occupation_code_detailed,occupation_code_major,0,0,0
2,household_family_status,household_summary,1,279,0
3,state_prev_residence,region_prev_residence,2,1974,0


Industry and occupation match exactly. Household `In group quarters` has multiple summaries (279 records), as do previous-state `Unknown` and `Abroad` (1,974 records). All remaining mappings match.

These are dataset mappings, not verified geography: the data pairs `New York` with `West`.

## 3. Which dimensions go into BUC?

Use 12 columns with relatively few values. Summary columns keep the cube smaller than their detailed versions. Exclude `instance_weight`; keep income as the AOI class label.

In [4]:
dimensions = [
    "year", "sex", "race", "education", "marital_status", "employment_status",
    "class_of_worker", "industry_code_major", "occupation_code_major",
    "tax_filer_status", "household_summary", "region_prev_residence",
]

df[dimensions].nunique().rename("distinct_values").to_frame()

,distinct_values
year,2
sex,2
race,5
education,17
marital_status,7
employment_status,8
class_of_worker,9
industry_code_major,24
occupation_code_major,15
tax_filer_status,6
